# 1. Azure Policy and Key Vault — Implementation

## Azure Policy

Azure Policy enforces rules on Azure resources. If a resource doesn't comply, the policy can **audit**, **deny**, **modify**, or **deploy** a remediation.

### Policy hierarchy

```
Policy definition → describes what to evaluate and what to do
Policy initiative → bundle of related definitions (e.g., "CIS Benchmark")
Policy assignment → bind definition/initiative to a scope (MG/Sub/RG)
```

### Key policy effects

| Effect | What happens | When to use |
|--------|-------------|-------------|
| **Audit** | Resource created, non-compliance logged | Monitoring / reporting |
| **Deny** | Resource creation blocked | Enforcement |
| **Modify** | Resource auto-corrected (e.g., add tag) | Auto-remediation |
| **DeployIfNotExists** | Deploy a resource if missing (e.g., enable diagnostics) | Auto-provisioning |
| **AuditIfNotExists** | Check if related resource exists | Monitoring |
| **Disabled** | Policy not evaluated | Temporarily turn off |

In [ ]:
import json

# Custom Azure Policy definition: require HTTPS on storage accounts
custom_policy = {
    'properties': {
        'displayName': 'Storage accounts must use HTTPS',
        'description': 'Deny creation of storage accounts that allow HTTP traffic.',
        'mode': 'All',
        'policyRule': {
            'if': {
                'allOf': [
                    {'field': 'type', 'equals': 'Microsoft.Storage/storageAccounts'},
                    {'field': 'Microsoft.Storage/storageAccounts/supportsHttpsTrafficOnly', 'notEquals': True},
                ]
            },
            'then': {
                'effect': 'deny'
            }
        },
    }
}

print('=== Custom Policy Definition ===\n')
print(json.dumps(custom_policy, indent=2))

print('\n--- Azure CLI commands ---')
print('# Create custom policy')
print('az policy definition create --name require-https-storage \\')
print('  --display-name "Storage accounts must use HTTPS" \\')
print('  --rules @policy-rule.json --mode All')
print()
print('# Assign to subscription')
print('az policy assignment create --name require-https \\')
print('  --policy require-https-storage \\')
print('  --scope /subscriptions/<sub-id>')
print()
print('# Check compliance')
print('az policy state list --policy-assignment require-https \\')
print('  --filter "complianceState eq \'NonCompliant\'"')

### Built-in initiatives for AZ-500

| Initiative | What it checks |
|-----------|----------------|
| **Microsoft Cloud Security Benchmark (MCSB)** | Azure-specific security best practices |
| **CIS Microsoft Azure Foundations Benchmark** | CIS-specific controls |
| **NIST SP 800-53** | US federal compliance |
| **ISO 27001** | International security standard |
| **PCI DSS** | Payment card industry |

```bash
# Assign MCSB initiative
az policy assignment create --name mcsb \
  --policy-set-definition /providers/Microsoft.Authorization/policySetDefinitions/1f3afdf9-d0c9-4c3d-847f-89da613e70a8 \
  --scope /subscriptions/<sub-id>
```

---
## Key Vault — deep implementation

### Access control: RBAC vs vault access policy

| | Vault access policy (legacy) | Azure RBAC (recommended) |
|-|---------------------------|-------------------------|
| **Granularity** | Per-vault | Per-key/secret/certificate |
| **Scope** | Vault level only | Can inherit from RG/subscription |
| **Condition support** | No | Yes (ABAC conditions) |
| **Audit** | Limited | Full Azure RBAC audit |

```bash
# Enable RBAC authorization (disable legacy access policies)
az keyvault update -n my-kv -g rg-prod --enable-rbac-authorization true
```

In [ ]:
# Key Vault security checklist
KV_CHECKLIST = [
    {'setting': 'RBAC authorization',           'cli': 'az keyvault update -n kv --enable-rbac-authorization true',          'why': 'Granular per-secret access control'},
    {'setting': 'Soft delete',                  'cli': 'az keyvault update -n kv --enable-soft-delete true',                'why': 'Recover accidentally deleted items (default: on)'},
    {'setting': 'Purge protection',             'cli': 'az keyvault update -n kv --enable-purge-protection true',           'why': 'Even admins can\'t permanently delete during retention'},
    {'setting': 'Private endpoint',             'cli': 'az network private-endpoint create ... --group-id vault',           'why': 'No public network access'},
    {'setting': 'Firewall (if no PE)',          'cli': 'az keyvault network-rule add -n kv --ip-address 203.0.113.0/24',    'why': 'Restrict to known IPs'},
    {'setting': 'Disable public access',        'cli': 'az keyvault update -n kv --public-network-access Disabled',         'why': 'Only accessible via private endpoint'},
    {'setting': 'Diagnostic logging',           'cli': 'az monitor diagnostic-settings create --resource kv ...',           'why': 'Audit who accessed what'},
    {'setting': 'Key rotation policy',          'cli': 'az keyvault key rotation-policy update --vault kv -n mykey ...',    'why': 'Auto-rotate keys on schedule'},
]

print('=== Key Vault Security Checklist ===\n')
for item in KV_CHECKLIST:
    print(f'☐ {item["setting"]}')
    print(f'  Why: {item["why"]}')
    print(f'  CLI: {item["cli"]}\n')

### Key rotation

```bash
# Set auto-rotation policy (rotate every 90 days, notify 30 days before expiry)
az keyvault key rotation-policy update --vault-name my-kv -n my-key \
  --value @rotation-policy.json
```

Rotation policy JSON:
```json
{
  "lifetimeActions": [
    {"trigger": {"timeAfterCreate": "P90D"}, "action": {"type": "Rotate"}},
    {"trigger": {"timeBeforeExpiry": "P30D"}, "action": {"type": "Notify"}}
  ],
  "attributes": {"expiryTime": "P1Y"}
}
```

### Backup and recovery

```bash
# Backup a secret
az keyvault secret backup --vault-name my-kv -n db-password --file db-password.bak

# Restore (same tenant, same geography)
az keyvault secret restore --vault-name my-kv-dr --file db-password.bak
```

**Exam tip**: Key Vault backups can only be restored to a vault in the **same tenant** and **same Azure geography**.

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **Azure Policy** | Effects: audit, deny, modify, deployIfNotExists. Initiatives bundle policies. |
| **MCSB** | Default compliance standard in Defender for Cloud. |
| **Key Vault RBAC** | Preferred over access policies. Per-secret granularity. |
| **Key rotation** | Auto-rotate with policies. Notify before expiry. |
| **KV backup** | Same tenant + same geography only. |
| **KV hardening** | Soft delete + purge protection + PE + diagnostics. |

**Next**: [Notebook 2 — Defender for Cloud](02_defender_for_cloud.ipynb)